# 01 - Data Cleaning

... 

## Examine columns and do team check

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/teams_game_level.csv')

print(df.shape)
print(df['team'].unique())

for col in df.columns:
    print(col)

(232220, 111)
<StringArray>
['NYR', 'SJS', 'PIT', 'EDM', 'WPG', 'PHI', 'DAL', 'N.J', 'T.B', 'MIN', 'COL',
 'ANA', 'VGK', 'TOR', 'NYI', 'MTL', 'SEA', 'STL', 'FLA', 'BUF', 'UTA', 'LAK',
 'ARI', 'OTT', 'WSH', 'NJD', 'TBL', 'CBJ', 'CHI', 'NSH', 'CGY', 'BOS', 'DET',
 'VAN', 'CAR', 'S.J', 'ATL', 'L.A']
Length: 38, dtype: str
team
season
name
gameId
playerTeam
opposingTeam
home_or_away
gameDate
position
situation
xGoalsPercentage
corsiPercentage
fenwickPercentage
iceTime
xOnGoalFor
xGoalsFor
xReboundsFor
xFreezeFor
xPlayStoppedFor
xPlayContinuedInZoneFor
xPlayContinuedOutsideZoneFor
flurryAdjustedxGoalsFor
scoreVenueAdjustedxGoalsFor
flurryScoreVenueAdjustedxGoalsFor
shotsOnGoalFor
missedShotsFor
blockedShotAttemptsFor
shotAttemptsFor
goalsFor
reboundsFor
reboundGoalsFor
freezeFor
playStoppedFor
playContinuedInZoneFor
playContinuedOutsideZoneFor
savedShotsOnGoalFor
savedUnblockedShotAttemptsFor
penaltiesFor
penalityMinutesFor
faceOffsWonFor
hitsFor
takeawaysFor
giveawaysFor
lowDangerShotsFor


In [2]:
# Check which seasons each code variant appears in
for pair in [('N.J', 'NJD'), ('T.B', 'TBL'), ('L.A', 'LAK'), ('S.J', 'SJS')]:
    old, new = pair
    old_seasons = sorted(df[df['team'] == old]['season'].unique())
    new_seasons = sorted(df[df['team'] == new]['season'].unique())
    print(f"{old}: {old_seasons}")
    print(f"{new}: {new_seasons}")
    print()

N.J: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
NJD: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

T.B: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
TBL: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

L.A: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
LAK: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

S.J: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(20

In [3]:
# Standardize team codes - fix formatting inconsistencies (pre 2021) and relocations
team_code_map = {
    'N.J': 'NJD',
    'T.B': 'TBL',
    'L.A': 'LAK',
    'S.J': 'SJS',
    'ATL': 'WPG',  # Atlanta Thrashers -> Winnipeg Jets (2011)
    'ARI': 'UTA',  # Arizona Coyotes -> Utah (2024)
}

df['team'] = df['team'].replace(team_code_map)
df['opposingTeam'] = df['opposingTeam'].replace(team_code_map)

print(df['team'].nunique(), 'unique teams after cleanup')
print(sorted(df['team'].unique()))

32 unique teams after cleanup
['ANA', 'BOS', 'BUF', 'CAR', 'CBJ', 'CGY', 'CHI', 'COL', 'DAL', 'DET', 'EDM', 'FLA', 'LAK', 'MIN', 'MTL', 'NJD', 'NSH', 'NYI', 'NYR', 'OTT', 'PHI', 'PIT', 'SEA', 'SJS', 'STL', 'TBL', 'TOR', 'UTA', 'VAN', 'VGK', 'WPG', 'WSH']


## Generate win/loss labels

`teams_game_level.csv` has no outcome column, and its own `home_or_away` field is
unreliable for ~115 games (both rows sometimes show 'AWAY'). Instead, we pull
`homeTeamWon` and `homeTeamCode` from the shots file - the same source `homeTeamWon`
was computed from - and derive win/loss from that directly. This also correctly
resolves shootout-decided games, which show equal `goalsFor`/`goalsAgainst` in the
team-game file and can't be resolved from goals alone.

In [4]:
# Filter to full-game totals (situation == 'all' sums every strength situation,
# giving one row per team per game with the true final score)
team_game_collapsed = df[df['situation'] == 'all'].copy()
print(team_game_collapsed.shape)

(46444, 111)


In [5]:
# Load shot data - used only to pull each game's outcome (homeTeamWon, homeTeamCode)
shots = pd.read_csv(
    '../data/raw/shots_2007-2024.csv',
    usecols=['game_id', 'homeTeamCode', 'homeTeamWon', 'season']
)
print(shots.shape)

(1960088, 4)


In [6]:
# shots' game_id lacks the season prefix and leading zero on game type
# (e.g. 20001 -> should be 2008020001). Reconstruct to match teams_game_level format.
shots['gameId'] = shots['season'] * 1_000_000 + shots['game_id']

game_outcomes = shots.drop_duplicates('gameId')[['gameId', 'homeTeamWon', 'homeTeamCode']].copy()

# homeTeamCode also needs the same team-code standardization applied earlier
game_outcomes['homeTeamCode'] = game_outcomes['homeTeamCode'].replace(team_code_map)

print(game_outcomes.shape)

(23142, 3)


In [7]:
team_game_collapsed = team_game_collapsed.merge(game_outcomes, on='gameId', how='left')
print(team_game_collapsed['homeTeamWon'].isna().sum(), 'unmatched rows out of', len(team_game_collapsed))

# Check which seasons the unmatched rows belong to
unmatched = team_game_collapsed[team_game_collapsed['homeTeamWon'].isna()]
print(unmatched['season'].value_counts().sort_index())

2790 unmatched rows out of 46444
season
2021       2
2025    2788
Name: count, dtype: int64


In [8]:
team_game_collapsed = team_game_collapsed[team_game_collapsed['homeTeamWon'].notna()].copy()
print(team_game_collapsed.shape)

(43654, 113)


In [9]:
# win = 1 if (this team is home AND home won) OR (this team is away AND home lost)
team_game_collapsed['win'] = (
    (team_game_collapsed['team'] == team_game_collapsed['homeTeamCode']) == (team_game_collapsed['homeTeamWon'] == 1)
).astype(int)

print(team_game_collapsed['win'].value_counts())

win
1    21827
0    21827
Name: count, dtype: int64


## Derive new variables

In [10]:
# Verify perspective: Team A's dZoneGiveawaysFor should equal Team B's 
# dZoneGiveawaysAgainst for the same game (situation == 'all', for a clean check)
check = df[df['situation'] == 'all'][['team', 'opposingTeam', 'gameId', 'dZoneGiveawaysFor', 'dZoneGiveawaysAgainst']]

sample_game = check['gameId'].iloc[0]
print(check[check['gameId'] == sample_game])

# Check units on iceTime
five_on_five = df[df['situation'] == '5on5'].copy()
five_on_five = five_on_five[five_on_five['iceTime'] > 0].copy()
print(five_on_five.shape)
print(five_on_five[['team', 'gameId', 'iceTime']].head(10))

      team opposingTeam      gameId  dZoneGiveawaysFor  dZoneGiveawaysAgainst
1      NYR          TBL  2008020001                4.0                    5.0
51651  TBL          NYR  2008020001                5.0                    4.0
(46420, 111)
   team      gameId  iceTime
2   NYR  2008020001   2283.0
7   NYR  2008020003   2791.0
12  NYR  2008020010   2854.0
17  NYR  2008020019   1743.0
22  NYR  2008020034   2695.0
27  NYR  2008020044   1508.0
32  NYR  2008020057   2548.0
37  NYR  2008020070   3112.0
42  NYR  2008020079   3007.0
47  NYR  2008020101   2288.0


In [11]:
# Build forecheck metrics on the 5-on-5 dataframe (all in one place, run once)

# Forecheck Turnover Index: dZoneGiveawaysAgainst per-60 (iceTime is in seconds)
five_on_five['forecheck_turnover_index'] = five_on_five['dZoneGiveawaysAgainst'] / five_on_five['iceTime'] * 3600
five_on_five['forecheck_turnover_index_z'] = (
    five_on_five['forecheck_turnover_index'] - five_on_five['forecheck_turnover_index'].mean()
) / five_on_five['forecheck_turnover_index'].std()

# Total Pressure Index: equal-weighted sum of standardized components
# (raw scales differ too much to sum directly - hits average ~21/game vs ~4.5 for 
# giveaways, so components are z-scored first to make them comparable)
for col in ['dZoneGiveawaysAgainst', 'takeawaysFor', 'hitsFor']:
    five_on_five[f'{col}_z'] = (five_on_five[col] - five_on_five[col].mean()) / five_on_five[col].std()

five_on_five['total_pressure_index_z'] = (
    five_on_five['dZoneGiveawaysAgainst_z'] + five_on_five['takeawaysFor_z'] + five_on_five['hitsFor_z']
)

print(five_on_five[[
    'team', 'opposingTeam', 'gameId',
    'forecheck_turnover_index', 'forecheck_turnover_index_z', 'total_pressure_index_z'
]].head())

print(five_on_five[['forecheck_turnover_index', 'forecheck_turnover_index_z', 'total_pressure_index_z']].isna().sum())

   team opposingTeam      gameId  forecheck_turnover_index  \
2   NYR          TBL  2008020001                  4.730618   
7   NYR          TBL  2008020003                  5.159441   
12  NYR          CHI  2008020010                  3.784163   
17  NYR          PHI  2008020019                  8.261618   
22  NYR          NJD  2008020034                  4.007421   

    forecheck_turnover_index_z  total_pressure_index_z  
2                    -0.204692                0.807612  
7                    -0.096186               -0.594064  
12                   -0.444174                0.575136  
17                    0.688759               -2.914085  
22                   -0.387682                2.770150  
forecheck_turnover_index      0
forecheck_turnover_index_z    0
total_pressure_index_z        0
dtype: int64


In [12]:
# Merge forecheck metrics onto the win/loss table
team_game_collapsed = team_game_collapsed.merge(
    five_on_five[['gameId', 'team', 'forecheck_turnover_index', 'forecheck_turnover_index_z', 'total_pressure_index_z']],
    on=['gameId', 'team'],
    how='left'
)

print(team_game_collapsed.shape)
print(team_game_collapsed['forecheck_turnover_index'].isna().sum(), 'unmatched rows')

# Check what's causing the unmatched rows
unmatched = team_game_collapsed[team_game_collapsed['forecheck_turnover_index'].isna()]
print(unmatched[['team', 'opposingTeam', 'gameId', 'gameDate']])

# Confirmed: these are the same 11 games excluded earlier for having iceTime == 0 
team_game_collapsed = team_game_collapsed[team_game_collapsed['forecheck_turnover_index'].notna()].copy()
print(team_game_collapsed.shape)

(43654, 117)
22 unmatched rows
      team opposingTeam      gameId  gameDate
640    NYR          FLA  2016020272  20161120
3134   PIT          CAR  2008030311  20090518
3929   EDM          UTA  2016020312  20161125
5787   WPG          MIN  2017030162  20180413
7376   DAL          NYI  2009020658  20100108
7998   DAL          COL  2017020675  20180113
12217  MIN          WPG  2017030162  20180413
12915  COL          SJS  2016020701  20170121
12995  COL          DAL  2017020675  20180113
17328  NYI          DAL  2009020658  20100108
22461  FLA          NYR  2016020272  20161120
22575  FLA          DET  2017020796  20180203
22723  FLA          CAR  2019020559  20191221
25649  UTA          EDM  2016020312  20161125
25705  UTA          WSH  2016021112  20170325
28357  WSH          UTA  2016021112  20170325
29990  CBJ          SJS  2009020885  20100210
37725  DET          FLA  2017020796  20180203
40689  CAR          FLA  2019020559  20191221
41120  CAR          PIT  2008030311  20090518
413

## Final sanity checks

In [13]:
# Check for any unexpected NaNs across the whole table
print(team_game_collapsed.isna().sum().sort_values(ascending=False).head(10))

# Spot-check a few rows to eyeball that everything lines up
print(team_game_collapsed[[
    'team', 'opposingTeam', 'gameId', 'gameDate', 'win', 
    'forecheck_turnover_index', 'total_pressure_index_z'
]].sample(5))

team                                 0
blockedShotAttemptsAgainst           0
penalityMinutesAgainst               0
penaltiesAgainst                     0
savedUnblockedShotAttemptsAgainst    0
savedShotsOnGoalAgainst              0
playContinuedOutsideZoneAgainst      0
playContinuedInZoneAgainst           0
playStoppedAgainst                   0
freezeAgainst                        0
dtype: int64
      team opposingTeam      gameId  gameDate  win  forecheck_turnover_index  \
13203  COL          MIN  2020020173  20210224    0                  0.000000   
17524  NYI          PIT  2011021157  20120329    1                  2.953240   
16269  TOR          BOS  2014020107  20141025    0                  2.612482   
4016   EDM          CGY  2017020407  20171202    1                 13.889863   
17450  NYI          TBL  2011020038  20111013    1                  2.500868   

       total_pressure_index_z  
13203               -3.167435  
17524               -0.840735  
16269               

## Save

In [14]:
team_game_collapsed.to_csv('../data/processed/team_game_collapsed.csv', index=False)
print('Saved:', team_game_collapsed.shape)

Saved: (43632, 117)
